# Geometry-of-Truth-style factual positive control

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JeffVallyath/geometry-of-truth/blob/v1.1.1/notebooks/geometry_of_truth.ipynb)

## Scope and current status

The project tests whether a language model encodes how a consideration bears on an action in a particular situation. This notebook covers the factual positive control for the activation method. The later Llama moral-relation development test has now passed, while the human-audited confirmatory test and rephrasing-flip prediction remain open.

The control asks whether the extraction and probing pipeline can recover factual truth when the answer mapping changes. Training and testing both include standard and reversed A/B mappings. A held-out transfer replaces A/B with 1/2. This tests whether the signal follows the factual class across answer symbols.

Layer 14 gives T=1.92 with a 95 percent bootstrap interval from 1.89 to 1.96. Directional consensus is C=0.999. None of 1,000 group-preserving Monte Carlo null runs match the observed statistic, giving p=1/1001 with the prespecified add-one correction. The 1/2 transfer gives T=1.84 with a 95 percent interval from 1.81 to 1.87.

## Construct boundary

This control matches the answer-mapping and activation-extraction mechanics used in the moral experiment. Factual truth is easier and represents a different target from moral endorsement. A pass establishes instrument sensitivity, while the checkerboard experiments supply the construct-specific evidence.

## Contents

1. Experimental population and split
2. Preserved first failure
3. Eight-partition direction ensemble
4. All-layer development selection
5. Confirmatory A/B result and 1/2 transfer
6. Source lineage and reproduction modes

## Start here

From GitHub, click the Open in Colab badge above. In Colab, choose Runtime and Run all. DEMO verifies the retained result files on CPU. ANALYSIS recomputes statistics from the hash-bound activation cache and is maintainer-only until that cache is published. FULL downloads the pinned model and datasets, extracts activations again, and requires an accepted model license, an HF_TOKEN Colab secret, and a CUDA GPU with at least 23,000 MiB of free memory.

| Mode | Public input | Typical resource | Output |
| --- | --- | --- | --- |
| DEMO | Aggregate result bundle | Colab CPU, minutes | Verified tables and figures |
| ANALYSIS | Retained activation cache and exact analysis lock | CPU plus cache storage | Recomputed statistics |
| FULL | Model and factual datasets | CUDA GPU, long run | New activations and statistics |

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PUBLIC_REPOSITORY = 'https://github.com/JeffVallyath/geometry-of-truth.git'
PUBLIC_REF = 'v1.1.1'
PUBLIC_COMMIT = 'cf605a169eef6cbe24ead242e0a5a39097df4f0d'
RUN_MODE = 'DEMO'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_ROOT = Path("/content/geometry-of-truth")
    if (REPO_ROOT / ".git").is_dir():
        origin = subprocess.run(["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"], check=True, text=True, capture_output=True).stdout.strip()
        if origin.rstrip("/").removesuffix(".git") != PUBLIC_REPOSITORY.rstrip("/").removesuffix(".git"):
            raise RuntimeError("The existing checkout has an unexpected origin")
        dirty = subprocess.run(["git", "-C", str(REPO_ROOT), "status", "--porcelain"], check=True, text=True, capture_output=True).stdout
        if dirty:
            raise RuntimeError("The existing checkout contains modified or untracked files")
        subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", PUBLIC_COMMIT], check=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--detach", PUBLIC_COMMIT], check=True)
    elif not (REPO_ROOT / "pyproject.toml").is_file():
        if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
            raise RuntimeError("The Colab repository directory exists but is not a usable checkout")
        REPO_ROOT.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "init"], check=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "remote", "add", "origin", PUBLIC_REPOSITORY], check=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", PUBLIC_COMMIT], check=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--detach", "FETCH_HEAD"], check=True)
    head = subprocess.run(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], check=True, text=True, capture_output=True).stdout.strip()
    if head != PUBLIC_COMMIT:
        raise RuntimeError(f"Expected public commit {PUBLIC_COMMIT}, found {head}")
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next(path for path in candidates if (path / "pyproject.toml").is_file())

if RUN_MODE == "ANALYSIS":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "requirements-truth-analysis.txt")], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(REPO_ROOT), "--no-deps"], check=True)
elif RUN_MODE == "FULL":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "requirements-truth-reproduction.txt")], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(REPO_ROOT), "--no-deps"], check=True)
elif IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(REPO_ROOT)], check=True)

OUTPUT_ROOT = Path(os.environ.get("GEOMETRY_OUTPUT_ROOT", "/content/geometry-results" if IN_COLAB else str(REPO_ROOT / "geometry-results")))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SOURCE_ROOT = str(REPO_ROOT / "src")
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)

print({"mode": RUN_MODE, "output_root": str(OUTPUT_ROOT)})

In [ ]:
from IPython.display import display

from geometry_of_truth.truth.contracts import load_bundle, number_lineage, retained_environment
from geometry_of_truth.truth.plots import layer_selection, permutation_null
from geometry_of_truth.truth.results import (
    consensus,
    bootstrap_intervals,
    design_counts,
    direction_method,
    mapping_checks,
    partition_effects,
    permutation_summary,
    prompt_examples,
    transfer,
    v1_diagnostics,
)

bundle = load_bundle(REPO_ROOT)
v1 = bundle["v1"]
v2 = bundle["v2"]
print("Artifact integrity verified")

## Experimental setup

The model is Meta Llama 3.1 8B Instruct. For each prompt, the extractor records every transformer layer at the last nonpadding token of the rendered chat prompt, immediately before the answer candidate begins. The prompt includes the mapping instruction and generation prompt, so final question token would be too narrow.

The factual sources contribute 1,496 affirmative city records and 1,496 matched negated records to each answer scheme. The experiment forms 748 proposition groups and keeps every linked version in one split, preventing an affirmative form from training a probe that later sees its negated partner in testing. Groups stay intact.

Each answer scheme contains 2,992 prompt records. Within each scheme, 1,796 records train the directions, 600 select the layer, and 596 remain untouched for the confirmatory test. The held-out 1 and 2 scheme mirrors the A and B design, producing 5,984 cached records across both schemes.

Standard prompts map true to A and false to B. Reversed prompts map true to B and false to A. The transfer prompts replace those symbols with 1 and 2. Every semantic class therefore appears under both answer assignments.

In [ ]:
display(prompt_examples())
display(design_counts(bundle["split"]))
display(mapping_checks(bundle["split"], v2))

## Frozen first attempt

The first design used the literal words True and False while instructing the model to reverse their meanings. AUROC measures ranking quality from zero to one, with 0.5 representing chance. Under the reversed instruction, semantic scoring reached 0.000349 while literal True minus False scoring reached 0.999651. The model followed the familiar words almost perfectly and ignored their assigned meanings almost perfectly.

That version remains a formal failure. The neutral-symbol design was specified afterward and keeps the failed result visible instead of rewriting the original rule.

In [ ]:
print(v1["frozen_v1_disposition"])
v1_table = v1_diagnostics(v1)
decisive_v1_rows = [
    "reversed instruction mapped macro AUROC",
    "reversed literal True minus False macro AUROC",
]
display(v1_table[v1_table["quantity"].isin(decisive_v1_rows)].reset_index(drop=True))

## Activation measure

The code divides the training groups into eight disjoint subsets. At each layer, it independently fits one direction on the rows inside each subset by subtracting the mean false activation from the mean true activation and normalizing the result. All eight fitted directions then score the same held-out development or test examples.

Each held-out activation is projected onto a subset direction, centered using that subset's training class means, and divided by that subset's training projection standard deviation. One subset effect is the mean true score minus the mean false score, with the two factual sources weighted equally. T averages the eight held-out effects. T has no fixed upper bound, and a positive value means the training orientation transfers to held-out data.

Directional consensus C is the length of the average of the eight unit directions. It ranges from zero to one. A value near one means that disjoint training subsets recover nearly the same direction. The notebook calls this an eight-partition signed separation rather than conventional cross-fitting because each direction uses its own partition, not the other seven.

In [ ]:
display(direction_method())
display(consensus(v2))

## Layer selection

The development sweep evaluates all 32 transformer layers. The horizontal axis shows layer index and the vertical axis shows signed development separation for each answer mapping. A candidate layer must separate truth correctly under both standard and reversed A/B mappings. The selection score uses the weaker effect, so one easy mapping cannot determine the winner. Layer 14 maximizes that rule. Test data remains untouched until this choice is fixed.

In [ ]:
display(layer_selection(v2).figure)

## Confirmatory result

At layer 14, all eight held-out subset effects are positive and range from 1.876 to 1.955. Their average is T=1.924, meaning that true and false test projections differ by about 1.92 training-standardized projection units after equal weighting of the two factual sources. A 2,000-group bootstrap gives a 95 percent interval from 1.893 to 1.956.

The eight unit directions produce C=0.9987 and an implied mean pairwise cosine of 0.9970. Both quantities show that the direction changes very little across training subsets.

The null keeps groups intact and flips labels for complete proposition groups, preserving every linked affirmative and negated set. Every null run repeats the full 32-layer development selection before evaluating the test statistic. None of 1,000 Monte Carlo runs reach T=1.924. The prespecified add-one calculation gives p=1/1001. This p value measures incompatibility with the group-preserving null, while T measures separation size.

In [ ]:
primary_effects = partition_effects(v2)
primary_effects = primary_effects[
    (primary_effects["verbalizer"] == "A/B") & (primary_effects["mapping"] == "overall")
][["partition", "signed effect"]]
display(primary_effects.reset_index(drop=True))
permutation_table = permutation_summary(v2)
shown_fields = ["permutations", "observed T", "null values at least observed", "add one p"]
display(permutation_table[permutation_table["field"].isin(shown_fields)].reset_index(drop=True))
display(permutation_null(v2).figure)
display(bootstrap_intervals(v2))

## Symbol transfer

The selected layer and fitted procedure next score prompts that use 1 and 2 instead of A and B. The table reports standard mapping, reversed mapping, and their equal-weight average for both symbol schemes. Signed T retains the training orientation. Macro AUROC reports ranking quality, where 0.5 is chance and 1.0 is perfect ranking within each factual source.

For A/B, standard mapping gives T=2.03 and reversed mapping gives T=1.82, producing the T=1.92 average. For held-out 1/2, the corresponding values are T=1.99 and T=1.69, producing T=1.84. Its 2,000-group bootstrap interval runs from 1.807 to 1.874. Macro AUROC equals 1.0 in every row. The transfer preserves separation after removing the original answer tokens.

In [ ]:
display(transfer(v2))

## Where every headline number comes from

Every displayed result comes from the public result bundle loaded at the start of the notebook. Integrity verification runs before any result table. The final two columns below name the data file and exact field or calculation for each headline quantity, allowing a reader to trace the prose back to stored values.

In [ ]:
display(number_lineage(bundle))

## Inherited method and project modifications

The experiment inherits the factual truth-geometry motivation, cities data family, and mean-difference direction from Marks and colleagues, *The Geometry of Truth* [arXiv 2310.06824](https://arxiv.org/abs/2310.06824) and its [source repository](https://github.com/saprmarks/geometry-of-truth).

| Component | This implementation |
| --- | --- |
| Answer surface | Neutral A/B mappings with a held-out 1/2 transfer |
| Grouping | Affirmative, negated, mapping-linked forms remain together |
| Estimator | Eight directions fitted on eight disjoint training subsets |
| Selection | All 32 layers selected on development data |
| Null | Group-sign Monte Carlo null repeats layer selection |
| Confirmation | Held-out test plus 2,000-group bootstrap intervals |

## Reproduction modes

DEMO verifies and presents the retained result files. ANALYSIS recomputes every statistic from a retained activation cache after checking every cache part. The cache is not publicly distributed yet, so this middle tier remains maintainer-only. FULL requests the model at the exact frozen revision, extracts activations, runs the analysis, and compares the reproduced measurements with the public reference.

ANALYSIS installs `requirements-truth-analysis.txt` before importing the numerical stack, then reads the cache location from TRUTH_CACHE_ROOT. FULL installs the separate GPU reproduction lock at the same point. All generated files use OUTPUT_ROOT. Set GEOMETRY_OUTPUT_ROOT to a mounted Google Drive directory before the setup cell when a run must survive a Colab reset. The following table records the software versions used for the retained run.

In [ ]:
display(retained_environment(bundle))

In [ ]:
if RUN_MODE == "ANALYSIS":
    from geometry_of_truth.truth.reproduce import reproduce_analysis

    cache_root = os.environ.get("TRUTH_CACHE_ROOT")
    if not cache_root:
        raise RuntimeError("Set TRUTH_CACHE_ROOT to the retained activation cache")
    analysis_run = reproduce_analysis(cache_root, OUTPUT_ROOT / "truth-analysis")
    display(analysis_run["comparison"])
else:
    print("Analysis reconstruction skipped")

In [ ]:
if RUN_MODE == "FULL":

    if IN_COLAB:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            os.environ["HF_TOKEN"] = token
    if not os.environ.get("HF_TOKEN"):
        raise RuntimeError("Add HF_TOKEN to Colab secrets and enable notebook access")
    from geometry_of_truth.truth.reproduce import reproduce_full; full_run = reproduce_full(OUTPUT_ROOT / "truth-reproduction"); display(full_run["checks"])
else:
    print("Set RUN_MODE to FULL and rerun from the first cell for raw reconstruction")

## Interpretation

The control establishes that the extraction and eight-partition direction ensemble can recover a known semantic distinction across two answer vocabularies. The completed moral-relation development result therefore rests on a pipeline that already passed a factual positive control. The open tests now ask whether the relation survives human-audited checkerboards and whether the original geometry predicts rephrasing-induced answer flips beyond text and native confidence.